# Python Bootcamp - Day 13 
## Jupyter Notebook Introduction
### OEE Data Exploration

Exploring production data loaded from `sample_data.csv`. 
Notebook demonstrates cell-based execution, inline output, and data storytelling.

In [1]:
import csv
from datetime import datetime

# Notebook metadata
print(f"Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Libraries loaded successfully")

Notebook initialized: 2026-05-27 11:02
Libraries loaded successfully


In [2]:
def load_csv(filepath):
    """Load production CSV into list of dictionaries."""
    records = []
    with open(filepath, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            records.append({
                "date":        row["date"],
                "machine":     row["machine"],
                "shift":       row["shift"],
                "actual_run":  int(row["actual_run"]),
                "planned":     int(row["planned"]),
                "units":       int(row["units"]),
                "good_units":  int(row["good_units"]),
                "defect_code": row["defect_code"]
            })
    return records

records = load_csv("sample_data.csv")
print(f"Records loaded: {len(records)}")
print(f"First record  : {records[0]}")

Records loaded: 240
First record  : {'date': '2024-01-01', 'machine': 'CNC Mill #4', 'shift': 'Day', 'actual_run': 461, 'planned': 480, 'units': 707, 'good_units': 707, 'defect_code': 'DEF-007'}


## OEE Calculations

Calculating Availability, Performance, and Quality for each record.  
OEE = Availability × Performance × Quality

In [3]:
def calc_oee(r):
    """Calculate OEE from a production record dictionary."""
    avail = r["actual_run"] / r["planned"]
    perf  = r["units"] / (r["planned"] * 1.8)
    qual  = r["good_units"] / r["units"]
    return avail * perf * qual

# Add OEE to each record
for r in records:
    r["oee"] = calc_oee(r)

print(f"OEE calculated for {len(records)} records")
print(f"Sample OEE values: {[round(r['oee'], 3) for r in records[:5]]}")

OEE calculated for 240 records
Sample OEE values: [0.786, 0.753, 0.76, 0.614, 0.895]


## Fleet Summary

Average OEE by machine across all shifts and dates.

In [4]:
# Group by machine
machine_groups = {}
for r in records:
    m = r["machine"]
    if m not in machine_groups:
        machine_groups[m] = []
    machine_groups[m].append(r["oee"])

# Summary table
print(f"{'Machine':<24} {'Avg OEE':>8}  {'Records':>8}  Status")
print("-" * 58)

for machine, oee_list in machine_groups.items():
    avg = sum(oee_list) / len(oee_list)
    status = "World Class" if avg >= 0.85 else "Acceptable" if avg >= 0.70 else "Below Threshold"
    print(f"{machine:<24} {avg*100:>7.1f}%  {len(oee_list):>8}  {status}")

Machine                   Avg OEE   Records  Status
----------------------------------------------------------
CNC Mill #4                 80.5%        60  Acceptable
Press #2                    80.2%        60  Acceptable
Assembly Line A             80.9%        60  Acceptable
Weld Station B              80.6%        60  Acceptable


## Key Findings

- Dataset spans **30 days** across **4 machines** and **2 shifts**
- Total records analyzed: **240**
- Fleet average OEE calculated per machine center

*Next notebook: defect trend analysis using control chart logic*